# Phase 3b — Final FCM Model
## FCM Gradational Lithofacies Classification
**Author:** Kumar Yuvraj (23GG5PE02) | IIT Kharagpur

Using the optimal `c` from Phase 3a, this notebook:
1. Fits the final FCM model on the full scaled feature matrix
2. Extracts the membership matrix `u` and hard labels
3. Inverse-transforms centroids back to log units for geological interpretation
4. Plots a depth-track membership heatmap revealing transition zones

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import joblib
import skfuzzy as fuzz
os.chdir('D:/Projects/fcm-lithofacies')

warnings.filterwarnings('ignore')

OUT_DATA = './outputs/data/'
OUT_FIGS = './outputs/figures/'

# ── Load cluster selection result from Phase 3a ────────────────────────────
with open(os.path.join(OUT_DATA, 'cluster_selection.json')) as f:
    sel = json.load(f)

C_OPTIMAL = sel['C_OPTIMAL']   # change here if you want to override
M_FUZZ    = sel['M_FUZZ']
ERROR     = 0.005
MAXITER   = 1000

print(f'Using C = {C_OPTIMAL},  m = {M_FUZZ}')

# ── Load data ──────────────────────────────────────────────────────────────
X_scaled = np.load(os.path.join(OUT_DATA, 'X_scaled.npy'))
scaler   = joblib.load(os.path.join(OUT_DATA, 'scaler.pkl'))
master   = pd.read_csv(os.path.join(OUT_DATA, 'force2020_preprocessed.csv'),
                       low_memory=False)

FEAT_COLS = ['GR', 'RHOB', 'NPHI', 'DTC', 'RDEP_LOG']
LOG_UNITS = ['API', 'g/cc', 'frac', 'μs/ft', 'log(Ω·m)']

print(f'X_scaled shape: {X_scaled.shape}')
print(f'master shape  : {master.shape}')

Using C = 7,  m = 2.0
X_scaled shape: (1127735, 5)
master shape  : (1127735, 9)


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Fit final FCM model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY on the full dataset (not the sub-sample used in Phase 3a):
# Cluster selection only needs a representative sub-sample for stable
# PC/FPE curves.  The final model must be fit on all data so that every
# depth sample in every well gets a membership vector — this is what
# Phase 4 uses to compute Vsh.

# NOTE: This will take 5–20 minutes depending on dataset size and hardware.
# Do not interrupt.

print(f'Fitting FCM: c={C_OPTIMAL}, N={X_scaled.shape[0]:,}  …')
print('(This may take 10–20 minutes on the full 1.17M row dataset)')

np.random.seed(42)
X_input = X_scaled.T   # skfuzzy: (n_features, n_samples)

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    X_input,
    c=C_OPTIMAL,
    m=M_FUZZ,
    error=ERROR,
    maxiter=MAXITER,
    init=None,
)

# u shape: (C_OPTIMAL, N_samples) — each column sums to 1.0
hard_labels = np.argmax(u, axis=0)   # shape (N,)

print(f'✓ FCM converged.  Final FPC = {fpc:.4f}')
print(f'  u shape      : {u.shape}  (clusters × samples)')
print(f'  cntr shape   : {cntr.shape}  (clusters × features)')
print(f'  Column sum check (first 5): {u[:, :5].sum(axis=0)}')

# Save membership matrix and hard labels
np.save(os.path.join(OUT_DATA, 'fcm_membership_matrix.npy'), u)
np.save(os.path.join(OUT_DATA, 'fcm_hard_labels.npy'), hard_labels)
np.save(os.path.join(OUT_DATA, 'fcm_centroids.npy'), cntr)
print('✓ u, hard_labels, cntr saved')

# Cluster size distribution
print('\nCluster size distribution:')
for cid in range(C_OPTIMAL):
    n = int((hard_labels == cid).sum())
    print(f'  Cluster {cid}: {n:>8,} samples  ({100.*n/len(hard_labels):.1f}%)')

Fitting FCM: c=7, N=1,127,735  …
(This may take 10–20 minutes on the full 1.17M row dataset)
✓ FCM converged.  Final FPC = 0.2844
  u shape      : (7, 1127735)  (clusters × samples)
  cntr shape   : (7, 5)  (clusters × features)
  Column sum check (first 5): [1. 1. 1. 1. 1.]
✓ u, hard_labels, cntr saved

Cluster size distribution:
  Cluster 0:  157,768 samples  (14.0%)
  Cluster 1:  147,686 samples  (13.1%)
  Cluster 2:  171,468 samples  (15.2%)
  Cluster 3:  163,273 samples  (14.5%)
  Cluster 4:  152,470 samples  (13.5%)
  Cluster 5:  169,238 samples  (15.0%)
  Cluster 6:  165,832 samples  (14.7%)


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Centroid interpretation table
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY inverse-transform:
# Centroids in scaled space (z-scores) are uninterpretable to a petrophysicist.
# Inverting back to original log units lets us say: "Cluster 3 has GR=28 API
# → clean sandstone" rather than "Cluster 3 has GR_z = −1.4".

cntr_original = scaler.inverse_transform(cntr)   # shape (C, 5)

# For RDEP, invert the log10 transform to get Ω·m
feat_display = list(FEAT_COLS)
units_display = list(LOG_UNITS)

centroid_df = pd.DataFrame(
    cntr_original,
    columns=[f'{f} ({u})' for f, u in zip(feat_display, units_display)]
)
centroid_df.index = [f'Cluster {i}' for i in range(C_OPTIMAL)]

# Add RDEP in Ω·m (un-log-transformed)
centroid_df['RDEP_OHM'] = 10 ** centroid_df[f'RDEP_LOG (log(Ω·m))']

print('Centroid values in original log units:')
print(centroid_df.round(2).to_string())

print('\n' + '─'*70)
print('PETROPHYSICAL INTERPRETATION (assign labels based on centroid values):')
print('─'*70)
print('''
Guidelines:
  Low GR (<40 API) + moderate RHOB (2.2–2.6) + low NPHI (<0.15)  → CLEAN SANDSTONE
  Low-mod GR + low RHOB (<2.1) + high NPHI (>0.3)                → GAS SAND (check RDEP)
  Mod GR (50–80) + mod RHOB + mod NPHI                            → SILT / TRANS SAND
  High GR (>80) + low RHOB (<2.3) + high NPHI (>0.3)             → SHALE
  Very low GR (<25) + very low RHOB (<2.0) + high NPHI           → COAL
  Low GR + high RHOB (>2.6) + low NPHI                            → CARBONATE
  High GR + high RDEP                                             → MARL / MARLY SHALE

Fill in your interpretation in the dict below after reviewing the table:
''')

# ── UPDATE THIS DICT after reviewing centroid table ────────────────────────
# Example (placeholder — replace with your actual centroid analysis):
CLUSTER_LABELS = {
    i: f'Cluster_{i}_LABEL_HERE' for i in range(C_OPTIMAL)
}
# Shale-type clusters (used for Vsh computation in Phase 4):
# SHALE_CLUSTERS = [2, 5]   ← update after reviewing table above

import json
with open(os.path.join(OUT_DATA, 'cluster_labels.json'), 'w') as f:
    json.dump({'cluster_labels': CLUSTER_LABELS,
               'C_OPTIMAL': C_OPTIMAL,
               'note': 'Update SHALE_CLUSTERS before running Phase 4'}, f, indent=2)
print(f'✓ cluster_labels.json saved  (update SHALE_CLUSTERS before Phase 4!)')

Centroid values in original log units:
           GR (API)  RHOB (g/cc)  NPHI (frac)  DTC (μs/ft)  RDEP_LOG (log(Ω·m))  RDEP_OHM
Cluster 0     67.10         2.38         0.32       100.62                 0.21      1.63
Cluster 1     45.30         2.48         0.20        82.21                 0.48      3.04
Cluster 2     52.82         2.01         0.50       144.73                -0.02      0.96
Cluster 3     66.22         2.20         0.41       126.48                 0.05      1.13
Cluster 4     88.95         2.55         0.26        82.52                 0.76      5.73
Cluster 5     80.07         2.04         0.49       144.07                 0.02      1.04
Cluster 6     91.52         2.42         0.35       101.32                 0.29      1.94

──────────────────────────────────────────────────────────────────────
PETROPHYSICAL INTERPRETATION (assign labels based on centroid values):
──────────────────────────────────────────────────────────────────────

Guidelines:
  Low GR (<40 

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Membership heatmap for representative well
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY a heatmap:
# A 1D depth track for one cluster misses the point.  The heatmap shows ALL
# c membership probabilities simultaneously for every depth interval.
# Transition zones appear as horizontal bands where NO single cluster
# has probability > 0.6 — the colour is spread across multiple columns.
# These are the scientifically interesting intervals — the gradational
# contacts that hard classifiers force into one category.

# Find representative well (most rows)
best_well = master.groupby('WELL').size().idxmax()
well_mask = master['WELL'].values == best_well

u_well    = u[:, well_mask]    # shape (C, N_well)
depth_well = master.loc[well_mask, 'DEPTH_MD'].values

print(f'Plotting heatmap for well: {best_well}  ({well_mask.sum():,} depth samples)')

# Sub-sample depth for readability (max 3000 rows)
if len(depth_well) > 3000:
    step = len(depth_well) // 3000
    depth_well = depth_well[::step]
    u_well     = u_well[:, ::step]

fig, (ax_heat, ax_max) = plt.subplots(1, 2, figsize=(12, 14),
                                        gridspec_kw={'width_ratios': [4, 1]})

# ── Heatmap: x = cluster index, y = depth, colour = membership ─────────
im = ax_heat.imshow(
    u_well.T,                     # (N_well, C)
    aspect='auto',
    cmap='YlOrRd',
    vmin=0, vmax=1,
    interpolation='nearest',
)
ax_heat.set_xlabel('Cluster index', fontsize=11)
ax_heat.set_ylabel('Depth index (shallowest = top)', fontsize=11)
ax_heat.set_xticks(range(C_OPTIMAL))
ax_heat.set_xticklabels([f'C{i}' for i in range(C_OPTIMAL)], fontsize=9)
ax_heat.set_title(
    f'FCM Membership Heatmap\nWell: {best_well}',
    fontsize=11, fontweight='bold'
)
plt.colorbar(im, ax=ax_heat, label='Membership probability', shrink=0.6)

# Annotate transition zones (max membership < 0.6)
max_mem = u_well.max(axis=0)   # shape (N_well,)
trans_mask = max_mem < 0.6
for i, is_trans in enumerate(trans_mask):
    if is_trans:
        ax_heat.axhline(i, color='cyan', lw=0.3, alpha=0.4)

# ── Right panel: max membership per depth ──────────────────────────────
ax_max.plot(max_mem, np.arange(len(max_mem)), color='steelblue', lw=1)
ax_max.fill_betweenx(np.arange(len(max_mem)), 0, max_mem,
                     where=(max_mem < 0.6),
                     color='cyan', alpha=0.4, label='Transition zone')
ax_max.axvline(0.6, color='red', ls='--', lw=1.2)
ax_max.set_xlim(0, 1.05)
ax_max.set_xlabel('Max cluster\nmembership', fontsize=9)
ax_max.set_title('Confidence', fontsize=9, fontweight='bold')
ax_max.legend(fontsize=8, loc='upper left')
ax_max.grid(axis='x', alpha=0.3)
ax_max.tick_params(labelleft=False)

pct_trans = 100. * trans_mask.sum() / len(trans_mask)
ax_max.text(0.05, 0.02,
            f'{pct_trans:.1f}%\ntransition',
            transform=ax_max.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', fc='lightyellow', ec='orange'))

plt.suptitle(
    f'Transition zones (max membership < 0.6): {pct_trans:.1f}% of depth intervals\n'
    f'— these are the gradational contacts FCM captures that hard classifiers miss',
    fontsize=10, y=1.01
)
plt.tight_layout()

safe_well = best_well.replace('/', '_').replace(' ', '_')
fig_path  = os.path.join(OUT_FIGS, f'membership_heatmap_{safe_well}.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'✓ Saved: {fig_path}')
print(f'\nTransition zone fraction (well {best_well}): {pct_trans:.1f}%')
print('→ This is the key FCM insight.')

Plotting heatmap for well: 25/2-7  (25,131 depth samples)
✓ Saved: ./outputs/figures/membership_heatmap_25_2-7.png

Transition zone fraction (well 25/2-7): 72.0%
→ This is the key FCM insight.
